# Figure 2 and Figure 3 panels

这一版用 UNI2 的 A2 best model，不用旧的 UNI 权重 `result_all_spatial/.../D_emph_L2_spatial_pancancer`。

- 权重：`data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt`。它是 `hce_group_fold_1.pt`，所以部署折是 **fold 1**，不是 fold 0。设定是 adjacent loss、L2→L1 的 NLL 权重 2、空间 k=8 均值、不带 gate。
- 划分：同一目录下的 `fold_plan.csv`。这一折留出 BRCA `rep2`、CRC `P2CRC`、GBM `P179161_Recurrent`。仓库根目录的 `data/PanCancer/fold_plan.csv` 是旧 UNI 的划分，CRC 的 fold 0 和 fold 1 对调，这里不用。
- 训练队列的预测：`data/PanCancer/pred_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42_deployment/`。同一个 best model 画了全部 896 张切片，包括训练时见过的切片。Figure 2b 只保留 fold 1，那才是这个权重没见过的切片。七个队列合在一起的 OOF L2 在 `2_Pred_statistic_visual_pancancer_uni2_ablation.ipynb`，不在这里。
- Figure 2c 的 HCC 示例仍是免疫治疗对照里的同几张切片。在这份划分里它们都不在 fold 1，所以空间图是部署预测，不是留出分数。
- Figure 3 用同一个权重在外部队列上的预测，目录名是 `pred_uni2_A2_bestmlp_paired_v2_seed42`。已经画好的空间 PNG 只有 NPC 完整；其余队列的 PNG 要等 `Pred_visual_*_uni2` 把图存进对应的 `figures/`。缺图的格留空，不回退到旧的 `pred_pancancer_section`。

图存在 `code/Hist2Pheno_pkg/dataset/figures_fig2_fig3/`。
裁图调用 `promoter/build_fig23.py`。留出分数在本 notebook 里按 UNI2 的 fold plan 重算，不再调用 `fold0_scores`。单核收到 spot 的捕获区调用 `Hist2Pheno_pkg/nuclei_to_spots.py`。


In [ ]:
from pathlib import Path
import csv
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code"))
sys.path.insert(0, str(REPO / "code" / "Hist2Pheno_pkg"))
sys.path.insert(0, str(REPO / "promoter"))

import build_fig23 as fig23
from plotting_palettes import get_palette
from uni_label_cv_helpers import macro_auc_ovr
import panel_figures as panels
from nuclei_to_spots import nuclei_to_spots

DATA = REPO / "data"
OUT = REPO / "code" / "Hist2Pheno_pkg" / "dataset" / "figures_fig2_fig3"
OUT.mkdir(parents=True, exist_ok=True)

# UNI2 A2 best model. best_mlp_gpu.pt is a copy of hce_group_fold_1.pt.
UNI2_TAG = "result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42"
UNI2_CKPT = (
    DATA / "PanCancer" / UNI2_TAG
    / "cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt"
)
UNI2_PRED = DATA / "PanCancer" / (
    "pred_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42_deployment"
)
EXT_PRED = "pred_uni2_A2_bestmlp_paired_v2_seed42"
FOLD_PLAN = DATA / "PanCancer" / UNI2_TAG / "fold_plan.csv"
BEST_FOLD = 1


def uni2_external_pred(cohort_dir, sample):
    """StarDist prediction written by the UNI2 A2 external launcher."""
    return DATA / cohort_dir / EXT_PRED / sample / f"{sample}_stardist_pred.h5ad"


def heldout_scores():
    """Macro-AUROC on the fold the deployed UNI2 A2 weight did not train on."""
    held = set()
    with open(FOLD_PLAN) as handle:
        for row in csv.DictReader(handle):
            if int(row["fold"]) == BEST_FOLD:
                held.add((row["dataset"], row["sample"]))
    order = ["Lung", "BRCA", "CRC", "HCC", "PDAC", "GIST", "GBM"]
    labels = {
        "Lung": "Lung", "BRCA": "Breast", "CRC": "CRC", "HCC": "HCC",
        "PDAC": "PDAC", "GIST": "GIST", "GBM": "GBM",
    }
    out = {d: {"L1": [], "L12": [], "L2": []} for d in order}
    with open(UNI2_PRED / "metrics_per_section.csv") as handle:
        for row in csv.DictReader(handle):
            if (row["dataset"], row["sample"]) not in held or not row["macro_auc_L2"]:
                continue
            for level in ("L1", "L12", "L2"):
                out[row["dataset"]][level].append(float(row[f"macro_auc_{level}"]))
    series = [(labels[d], out[d]) for d in order]
    print(f"UNI2 A2 best model, held-out fold {BEST_FOLD}")
    print(UNI2_CKPT)
    for name, levels in series:
        med = {k: (float(np.median(v)) if v else None) for k, v in levels.items()}
        print(
            f"  {name}: n={len(levels['L2'])} median L1/L12/L2 "
            f"{med['L1']:.3f} {med['L12']:.3f} {med['L2']:.3f}"
        )
    return series


def _matched_pred_h5ad(data_root, cancer, sample):
    return UNI2_PRED / "matched" / cancer / sample / f"{sample}_matched_pred.h5ad"


def _stardist_roc_dir(data_root, cancer, sample):
    return UNI2_PRED / "stardist_roc" / f"{cancer}__{sample}"


panels.matched_pred_h5ad = _matched_pred_h5ad
panels.stardist_roc_dir = _stardist_roc_dir

plt.rcParams.update({
    "font.family": "Liberation Sans",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.4,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

L2_PALETTE = get_palette("pancancer", "fine")
L12_PALETTE = get_palette("pancancer", "intermediate")
L1_PALETTE = get_palette("pancancer", "coarse")
TIER_PALETTE = {"L1": L1_PALETTE, "L12": L12_PALETTE, "L2": L2_PALETTE}

try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    def display(obj):
        print(obj)


def save_fig(fig, stem):
    png = OUT / f"{stem}.png"
    pdf = OUT / f"{stem}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(png)
    if Image is not None:
        display(Image(filename=str(png)))
    return png


## Pan-cancer color swatch

Same figure as `python code/Hist2Pheno_pkg/write_palette_swatch.py`, drawn by `plotting_palettes.write_pancancer_swatch`. Saved here as PNG and PDF.


In [ ]:
from plotting_palettes import write_pancancer_swatch

swatch = write_pancancer_swatch(OUT / "pancancer_L1_L12_L2_swatch.png")
print(swatch)
print(swatch.with_suffix(".pdf"))
if Image is not None:
    display(Image(filename=str(swatch)))


## Figure 2a. Section is the hold-out group

每个格子是该验证折留出的整张切片数。灰色列是 fold 1，也就是 UNI2 A2 `best_mlp` 的那一折。数字来自 `result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/fold_plan.csv`，共 896 张切片。CRC 三张都在交叉验证里：P1CRC 是 fold 0，P2CRC 是 fold 1，P5CRC 是 fold 2。


In [ ]:
ORDER = ["Lung", "BRCA", "CRC", "HCC", "PDAC", "GIST", "GBM"]
SHOW = {"BRCA": "Breast"}

fold = pd.read_csv(FOLD_PLAN)
fold["fold"] = fold["fold"].astype(int)
counts = (
    fold.groupby(["dataset", "fold"]).size()
    .unstack("fold")
    .reindex(index=ORDER, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)
counts.to_csv(OUT / "fig2a_section_counts.csv")
print(f"fold {BEST_FOLD} sections", int(counts[BEST_FOLD].sum()))
display(counts.rename(index=SHOW))

fig, ax = plt.subplots(figsize=(8.6 / 2.54, 6.8 / 2.54))
ax.set_xlim(-1.35, 4.7)
ax.set_ylim(len(ORDER) - 0.45, -1.25)
ax.axis("off")
ax.text(2, -1.05, "Validation fold", ha="center", va="center", fontsize=7)
for j, fold_id in enumerate(range(5)):
    ax.text(j, -0.62, str(fold_id), ha="center", va="center", fontsize=7,
            fontweight="bold" if fold_id == BEST_FOLD else "regular")
for i, dataset in enumerate(ORDER):
    ax.text(-0.55, i, SHOW.get(dataset, dataset), ha="right", va="center", fontsize=7)
    for j, fold_id in enumerate(range(5)):
        value = int(counts.loc[dataset, fold_id])
        if value == 0:
            continue
        ax.add_patch(plt.Rectangle(
            (j - 0.36, i - 0.34), 0.72, 0.68,
            facecolor="#E8E8E8" if fold_id == BEST_FOLD else "white",
            edgecolor="black", linewidth=0.5, zorder=2,
        ))
        ax.text(j, i, str(value), ha="center", va="center", fontsize=7, zorder=3)
save_fig(fig, "fig2a_section_split")


In [ ]:
section_counts = (
    fold.groupby(["dataset", "fold"]).size()
    .unstack("fold")
    .reindex(index=ORDER, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)
cell_counts = (
    fold.groupby(["dataset", "fold"])["n_keep"].sum()
    .unstack("fold")
    .reindex(index=ORDER, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)

n = len(ORDER)
sec_w, nuc_w = 0.78, 1.28
sec_x = np.arange(5, dtype=float)
nuc_x = 5.55 + np.arange(5) * 1.42
box_h = 0.68
label_x = -0.55

fig, ax = plt.subplots(figsize=(18.0 / 2.54, 7.6 / 2.54))
ax.set_xlim(-1.55, nuc_x[-1] + nuc_w / 2 + 0.12)
ax.set_ylim(n - 0.45, -1.35)
ax.axis("off")
ax.text(sec_x.mean(), -1.08, "Sections", ha="center", va="center", fontsize=7)
ax.text(nuc_x.mean(), -1.08, "Nuclei", ha="center", va="center", fontsize=7)
for xs in (sec_x, nuc_x):
    for j, fold_id in enumerate(range(5)):
        ax.text(
            xs[j], -0.58, str(fold_id), ha="center", va="center", fontsize=7,
            fontweight="bold" if fold_id == BEST_FOLD else "regular",
        )

def draw_block(table, xs, width, fmt):
    for i, dataset in enumerate(ORDER):
        for j, fold_id in enumerate(range(5)):
            value = int(table.loc[dataset, fold_id])
            if value == 0:
                continue
            ax.add_patch(plt.Rectangle(
                (xs[j] - width / 2, i - box_h / 2), width, box_h,
                facecolor="#E8E8E8" if fold_id == BEST_FOLD else "white",
                edgecolor="black", linewidth=0.5, zorder=2,
            ))
            ax.text(xs[j], i, fmt(value), ha="center", va="center", fontsize=7, zorder=3)

for i, dataset in enumerate(ORDER):
    ax.text(label_x, i, SHOW.get(dataset, dataset), ha="right", va="center", fontsize=7)
draw_block(section_counts, sec_x, sec_w, str)
draw_block(cell_counts, nuc_x, nuc_w, "{:,}".format)
save_fig(fig, "fig2a_sections_and_cells")



In [ ]:
# Stacked sections (top) and nuclei (bottom) in one box per fold.
# Lung is shown as PF only on this panel.
section_counts = (
    fold.groupby(["dataset", "fold"]).size()
    .unstack("fold")
    .reindex(index=ORDER, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)
cell_counts = (
    fold.groupby(["dataset", "fold"])["n_keep"].sum()
    .unstack("fold")
    .reindex(index=ORDER, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)

row_labels = dict(SHOW)
row_labels["Lung"] = "PF"

n = len(ORDER)
box_w, box_h = 1.18, 0.76
xs = np.arange(5, dtype=float) * 1.38
label_x = xs[0] - box_w / 2 - 0.18

# y increases downward (ylim is inverted). Smaller y is the top of the page.
SEC_FACE, NUC_FACE = "#D7E6F2", "#F4E4D4"
SEC_INK, NUC_INK = "#1B4F72", "#7A3E12"
EDGE, EDGE0 = "#4D4D4D", "#1A1A1A"

fig, ax = plt.subplots(figsize=(13.6 / 2.54, 8.4 / 2.54))
ax.set_xlim(label_x - 1.15, xs[-1] + box_w / 2 + 0.12)
ax.set_ylim(n - 0.42, -1.22)
ax.axis("off")

ax.text(xs[0], -1.05, "Fold", ha="center", va="center", fontsize=7)
ax.add_patch(plt.Rectangle(
    (xs[2] - 0.15, -1.13), 0.22, 0.16,
    facecolor=SEC_FACE, edgecolor=EDGE, linewidth=0.35, zorder=2,
))
ax.text(xs[2] + 0.14, -1.05, "sections", ha="left", va="center", fontsize=6, color=SEC_INK)
ax.add_patch(plt.Rectangle(
    (xs[3] + 0.15, -1.13), 0.22, 0.16,
    facecolor=NUC_FACE, edgecolor=EDGE, linewidth=0.35, zorder=2,
))
ax.text(xs[3] + 0.44, -1.05, "nuclei", ha="left", va="center", fontsize=6, color=NUC_INK)
for j, fold_id in enumerate(range(5)):
    ax.text(
        xs[j], -0.62, str(fold_id),
        ha="center", va="center", fontsize=7, color="#111111",
        fontweight="bold" if fold_id == BEST_FOLD else "regular",
    )

for i, dataset in enumerate(ORDER):
    ax.text(
        label_x, i, row_labels.get(dataset, dataset),
        ha="right", va="center", fontsize=7, color="#111111",
    )
    for j, fold_id in enumerate(range(5)):
        n_sec = int(section_counts.loc[dataset, fold_id])
        n_nuc = int(cell_counts.loc[dataset, fold_id])
        if n_sec == 0 and n_nuc == 0:
            continue
        x0 = xs[j] - box_w / 2
        y_top = i - box_h / 2
        ax.add_patch(plt.Rectangle(
            (x0, y_top), box_w, box_h / 2,
            facecolor=SEC_FACE, edgecolor="none", zorder=2,
        ))
        ax.add_patch(plt.Rectangle(
            (x0, i), box_w, box_h / 2,
            facecolor=NUC_FACE, edgecolor="none", zorder=2,
        ))
        ax.plot(
            [x0, x0 + box_w], [i, i],
            color="white", lw=0.6, solid_capstyle="butt", zorder=3,
        )
        ax.add_patch(plt.Rectangle(
            (x0, y_top), box_w, box_h,
            facecolor="none",
            edgecolor=EDGE0 if fold_id == BEST_FOLD else EDGE,
            linewidth=0.7 if fold_id == BEST_FOLD else 0.35,
            zorder=4,
        ))
        ax.text(
            xs[j], i - box_h / 4, str(n_sec),
            ha="center", va="center", fontsize=6.5, color=SEC_INK, zorder=5,
        )
        ax.text(
            xs[j], i + box_h / 4, f"{n_nuc:,}",
            ha="center", va="center", fontsize=6, color=NUC_INK, zorder=5,
        )

save_fig(fig, "fig2a_sections_nuclei_stacked")


## Figure 2a, continued. Kept cells and pathologist labels

上面把切片数和细胞核数画在同一张图里，行和折对齐。下面两张仍用同一套 5 折。

- 细胞数是 `fold_plan.csv` 的 `n_keep`：映射到 pan-cancer 标签后、这一折留出的细胞核数。
- 条形图是这些细胞核的病理标注，存在 `data/PanCancer/label_cache/`。`y_l1`、`y_l12`、`y` 分别是 L1、L12、L2。它们来自各队列单独训练时的标注，经 `celltype_pancancer` 映射后保留。条形是该折、该癌种内部的比例。纵轴上加粗的数字是部署折 fold 1。某一折没有这个癌种时，这一行留空。


In [ ]:

CACHE = DATA / "PanCancer/label_cache"
cell_counts = (
    fold.groupby(["dataset", "fold"])["n_keep"].sum()
    .unstack("fold")
    .reindex(index=ORDER, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)
cell_counts.to_csv(OUT / "fig2a_cell_counts.csv")
display(cell_counts.rename(index=SHOW))

fig, ax = plt.subplots(figsize=(14.2 / 2.54, 7.4 / 2.54))
ax.set_xlim(-1.7, 4.65)
ax.set_ylim(len(ORDER) - 0.45, -1.25)
ax.axis("off")
ax.text(2, -1.05, "Validation fold", ha="center", va="center", fontsize=7)
for j, fold_id in enumerate(range(5)):
    ax.text(j, -0.62, str(fold_id), ha="center", va="center", fontsize=7,
            fontweight="bold" if fold_id == BEST_FOLD else "regular")
for i, dataset in enumerate(ORDER):
    ax.text(-0.62, i, SHOW.get(dataset, dataset), ha="right", va="center", fontsize=7)
    for j, fold_id in enumerate(range(5)):
        value = int(cell_counts.loc[dataset, fold_id])
        if value == 0:
            continue
        ax.add_patch(plt.Rectangle(
            (j - 0.46, i - 0.34), 0.92, 0.68,
            facecolor="#E8E8E8" if fold_id == BEST_FOLD else "white",
            edgecolor="black", linewidth=0.5, zorder=2,
        ))
        ax.text(j, i, f"{value:,}", ha="center", va="center", fontsize=6, zorder=3)
save_fig(fig, "fig2a_cell_counts")

LEVEL_ARRAY = {"L1": "y_l1", "L12": "y_l12", "L2": "y"}
PALETTE = {"L1": L1_PALETTE, "L12": L12_PALETTE, "L2": L2_PALETTE}
from collections import Counter
gt_counts = {}
for rec in fold.itertuples(index=False):
    cache_path = CACHE / f"{rec.section_id.replace('::', '__')}.npz"
    z = np.load(cache_path, allow_pickle=True)
    for level, key in LEVEL_ARRAY.items():
        labels, n = np.unique(z[key], return_counts=True)
        bucket = gt_counts.setdefault((rec.dataset, int(rec.fold), level), Counter())
        for label, count in zip(labels, n):
            bucket[str(label)] += int(count)

rows = []
for (dataset, fold_id, level), bucket in gt_counts.items():
    total = sum(bucket.values())
    for label, count in bucket.items():
        rows.append({
            "dataset": dataset,
            "fold": fold_id,
            "level": level,
            "label": label,
            "n": count,
            "fraction": count / total,
        })
gt_table = pd.DataFrame(rows).sort_values(["level", "dataset", "fold", "label"])
gt_table.to_csv(OUT / "fig2a_gt_composition.csv", index=False)
print(f"kept cells in cache {int(gt_table.loc[gt_table.level == 'L2', 'n'].sum()):,}")

fig, axes = plt.subplots(
    len(ORDER), 3,
    figsize=(18.0 / 2.54, 16.2 / 2.54),
    sharex=True,
    gridspec_kw={"wspace": 0.18, "hspace": 0.28},
)
for i, dataset in enumerate(ORDER):
    for j, level in enumerate(("L1", "L12", "L2")):
        ax = axes[i, j]
        ax.axhspan(-0.46, 0.46, color="#F3F3F3", zorder=0)
        palette = PALETTE[level]
        present = [
            name for name in palette
            if any(gt_counts.get((dataset, fold_id, level), {}).get(name, 0) for fold_id in range(5))
        ]
        for fold_id in range(5):
            bucket = gt_counts.get((dataset, fold_id, level), {})
            total = sum(bucket.values())
            if total == 0:
                continue
            left = 0.0
            for name in present:
                count = bucket.get(name, 0)
                if count == 0:
                    continue
                width = count / total
                ax.barh(
                    fold_id, width, left=left, height=0.72,
                    color=palette[name], linewidth=0, zorder=2,
                )
                left += width
        ax.set_xlim(0, 1)
        ax.set_ylim(4.55, -0.55)
        ax.tick_params(length=2, width=0.4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if j == 0:
            ax.set_yticks(range(5))
            ax.set_yticklabels([str(k) for k in range(5)])
            for tick in ax.get_yticklabels():
                if tick.get_text() == str(BEST_FOLD):
                    tick.set_fontweight("bold")
            ax.set_ylabel(SHOW.get(dataset, dataset), fontsize=7)
        else:
            ax.set_yticks(range(5))
            ax.set_yticklabels([])
        if i == 0:
            ax.set_title(level, fontsize=7, pad=3)
        if i == len(ORDER) - 1:
            ax.set_xlabel("Fraction", fontsize=7)
        else:
            ax.set_xlabel("")
            ax.tick_params(labelbottom=False)

for j, level in enumerate(("L1", "L12", "L2")):
    palette = PALETTE[level]
    used = {
        row.label
        for row in gt_table.itertuples(index=False)
        if row.level == level
    }
    present = [name for name in palette if name in used]
    handles = [
        plt.Line2D(
            [0], [0], linestyle="none", marker="s", markersize=4,
            markerfacecolor=palette[name], markeredgecolor="none", label=name,
        )
        for name in present
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.20 + 0.30 * j, -0.01),
        frameon=False,
        fontsize=5.5,
        ncol=2 if level == "L1" else 3,
        columnspacing=0.6,
        handletextpad=0.25,
    )
fig.savefig(OUT / "fig2a_gt_composition.png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(OUT / "fig2a_gt_composition.pdf", bbox_inches="tight", facecolor="white")
plt.close(fig)
print(OUT / "fig2a_gt_composition.png")
if Image is not None:
    display(Image(filename=str(OUT / "fig2a_gt_composition.png")))


## Figure 2b. Held-out macro-AUROC

每个点是 fold 1 的一张切片，也就是 UNI2 A2 `best_mlp` 训练时没见过的切片。`heldout_scores` 把部署目录里的 `metrics_per_section.csv` 接到 UNI2 的 `fold_plan.csv`，只保留 `fold == 1`。CRC 在这一折留出的是 P2CRC。P5CRC 在 fold 2，划分图里有它，这张分数图不画。LungIncomplete 没有病理标注，也不画。横轴按这一折的 L2 中位数从高到低排列。黑竖线是四分位距，黑横线是中位数。

这是 matched-cell 的 macro-AUROC，和 `2_Pred_statistic_visual_pancancer_uni2_ablation.ipynb` 里七个队列合在一起的 OOF 不是同一个数。部署目录里另外 715 张切片被这个权重训练过，不进这张图。


In [ ]:
series = heldout_scores()
series = sorted(series, key=lambda item: float(np.median(item[1]["L2"])), reverse=True)
# Lighter L1 blue, L12 green, L2 red, still separable at the small marker size.
level_color = {"L1": "#8EADD4", "L12": "#7DCFC2", "L2": "#F3A99F"}
rows = []
for name, levels in series:
    n = len(levels["L2"])
    for level, values in levels.items():
        for value in values:
            rows.append({"cancer": name, "level": level, "macro_auroc": value, "n_sections": n})
points = pd.DataFrame(rows)
points.to_csv(OUT / "fig2b_bestfold_macro_auroc.csv", index=False)
summary = (
    points.groupby(["cancer", "level"])["macro_auroc"]
    .agg(n="size", median="median")
    .reset_index()
)
display(summary.pivot(index="cancer", columns="level", values="median").reindex(
    [name for name, _ in series]
)[["L1", "L12", "L2"]])

fig, ax = plt.subplots(figsize=(8 / 2.54, 6 / 2.54))
rng = np.random.default_rng(0)
levels = ("L1", "L12", "L2")
for i, (name, grouped) in enumerate(series):
    for j, level in enumerate(levels):
        vals = np.asarray(grouped[level], dtype=float)
        xx = i + (j - 1) * 0.26
        color = level_color[level]
        if len(vals) > 1:
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
            ax.plot([xx, xx], [q1, q3], color="#222222", lw=0.6, zorder=2, solid_capstyle="butt")
            jitter = rng.uniform(-0.06, 0.06, size=len(vals))
            ax.scatter(
                np.full(len(vals), xx) + jitter, vals,
                s=7, c=color, alpha=0.95, linewidths=0, zorder=3,
                label=level if i == 0 else None,
            )
            ax.plot([xx - 0.08, xx + 0.08], [med, med], color="#222222", lw=0.9, zorder=4, solid_capstyle="butt")
        else:
            ax.scatter(
                [xx], vals, s=16, c=color, linewidths=0.4, edgecolors="white",
                zorder=3, label=level if i == 0 else None,
            )
ax.set_xticks(range(len(series)))
ax.set_xticklabels([
    f"{'BRCA' if name == 'Breast' else name}\n(n={len(grouped['L2'])})"
    for name, grouped in series
])
ax.set_ylabel("Macro-AUROC")
ax.set_xlim(-0.55, len(series) - 0.45)
ax.set_ylim(0.45, 1.02)
ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax.yaxis.grid(True, color="#E6E6E6", lw=0.4, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
for spine in ax.spines.values():
    spine.set_color("#222222")
    spine.set_linewidth(0.4)
ax.tick_params(length=2.5, width=0.4, color="#222222")
ax.legend(
    frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.04),
    borderaxespad=0, handletextpad=0.3, columnspacing=1.0, markerscale=1.3,
)
fig.tight_layout()
save_fig(fig, "fig2b_bestfold_macro_auroc")


In [ ]:
# Figure 2a and 2b on one row. Rows follow the Figure 2b order:
# Best-fold (fold 1) L2 median, high at the top. The right panel is that strip plot
# with cancers on y and macro-AUROC on x, so each row lines up.
paired = series
row_order = ["BRCA" if name == "Breast" else name for name, _ in paired]
level_color = {"L1": "#8EADD4", "L12": "#7DCFC2", "L2": "#F3A99F"}

section_counts = (
    fold.groupby(["dataset", "fold"]).size()
    .unstack("fold")
    .reindex(index=row_order, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)
cell_counts = (
    fold.groupby(["dataset", "fold"])["n_keep"].sum()
    .unstack("fold")
    .reindex(index=row_order, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)

n = len(row_order)
sec_w, nuc_w = 0.78, 1.28
sec_x = np.arange(5, dtype=float)
nuc_x = 5.55 + np.arange(5) * 1.42
box_h = 0.68
label_x = -0.55
ymin, ymax = n - 0.45, -1.35

fig = plt.figure(figsize=(18.0 / 2.54, 8.2 / 2.54))
gs = fig.add_gridspec(1, 2, width_ratios=[2.15, 0.95], wspace=0.08)
ax = fig.add_subplot(gs[0, 0])
axr = fig.add_subplot(gs[0, 1])

ax.set_xlim(-1.55, nuc_x[-1] + nuc_w / 2 + 0.15)
ax.set_ylim(ymin, ymax)
ax.axis("off")
ax.text(sec_x.mean(), -1.08, "Sections", ha="center", va="center", fontsize=7)
ax.text(nuc_x.mean(), -1.08, "Nuclei", ha="center", va="center", fontsize=7)
for xs in (sec_x, nuc_x):
    for j, fold_id in enumerate(range(5)):
        ax.text(
            xs[j], -0.58, str(fold_id), ha="center", va="center", fontsize=7,
            fontweight="bold" if fold_id == BEST_FOLD else "regular",
        )

def draw_block(table, xs, width, fmt, size):
    for i, dataset in enumerate(row_order):
        for j, fold_id in enumerate(range(5)):
            value = int(table.loc[dataset, fold_id])
            if value == 0:
                continue
            ax.add_patch(plt.Rectangle(
                (xs[j] - width / 2, i - box_h / 2), width, box_h,
                facecolor="#E8E8E8" if fold_id == BEST_FOLD else "white",
                edgecolor="black", linewidth=0.5, zorder=2,
            ))
            ax.text(xs[j], i, fmt(value), ha="center", va="center", fontsize=size, zorder=3)

for i, dataset in enumerate(row_order):
    ax.text(label_x, i, dataset, ha="right", va="center", fontsize=7)
draw_block(section_counts, sec_x, sec_w, str, 7)
draw_block(cell_counts, nuc_x, nuc_w, "{:,}".format, 6)

axr.set_xlim(0.45, 1.05)
axr.set_ylim(ymin, ymax)
axr.axis("off")
axr.text(0.75, -1.08, "Macro-AUROC", ha="center", va="center", fontsize=7)
for tick in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
    axr.plot([tick, tick], [-0.15, ymin], color="#E6E6E6", lw=0.4, zorder=0)
    axr.text(tick, -0.58, f"{tick:.1f}", ha="center", va="center", fontsize=6, color="#222222")

rng = np.random.default_rng(0)
for i, (_name, grouped) in enumerate(paired):
    for j, level in enumerate(("L1", "L12", "L2")):
        vals = np.asarray(grouped[level], dtype=float)
        yy = i + (j - 1) * 0.20
        color = level_color[level]
        if len(vals) > 1:
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
            axr.plot([q1, q3], [yy, yy], color="#222222", lw=0.6, zorder=2, solid_capstyle="butt")
            jitter = rng.uniform(-0.045, 0.045, size=len(vals))
            axr.scatter(
                vals, np.full(len(vals), yy) + jitter,
                s=7, c=color, alpha=0.95, linewidths=0, zorder=3,
                label=level if i == 0 else None,
            )
            axr.plot([med, med], [yy - 0.07, yy + 0.07], color="#222222", lw=0.9, zorder=4, solid_capstyle="butt")
        else:
            axr.scatter(
                vals, [yy], s=16, c=color, linewidths=0.4, edgecolors="white",
                zorder=3, label=level if i == 0 else None,
            )
axr.legend(
    frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.42),
    handletextpad=0.3, borderaxespad=0, markerscale=1.3, fontsize=6,
)
save_fig(fig, "fig2ab_sections_nuclei_auroc")


## Figure 2c. HCC section from the immunotherapy contrast

`kqr-83230_aligned_3bba12d6` 在 UNI2 A2 的 fold plan 里是 fold 0。部署权重是 fold 1，训练时见过这张切片，所以下面的空间图和 AUROC 是部署预测，不是留出分数。留出分数只看 Figure 2b。接触表 PNG 早于现在的 `plotting_palettes.py`，所以这里不裁那张图，而是用 UNI2 部署目录里的 `matched_pred.h5ad` 重画。颜色是 `get_palette("pancancer")`：L1 coarse、L12 intermediate、L2 fine。每一列最下方是该列在标注或预测里出现过的类。

这张切片来自 Wu, Trevino 等, bioRxiv 2025.06.11.656869 的同一项 nivolumab ± ipilimumab 试验：病人 85560，无应答，治疗后（区域 73_4）。预测里肿瘤约 32%、巨噬细胞约 42%，对应文中 Fig. 5 的髓系富集肿瘤。


In [ ]:
import anndata as ad

hcc_sample = "kqr-83230_aligned_3bba12d6"
adata = ad.read_h5ad(panels.matched_pred_h5ad(DATA, "HCC", hcc_sample))

# Visual params (edit here)
POINT_S = 0.4
FIGSIZE_CM = (16.4, 12.6)

fig = panels.plot_gt_pred_spatial(
    adata,
    TIER_PALETTE,
    point_s=POINT_S,
    figsize_cm=FIGSIZE_CM,
)
save_fig(fig, "fig2c_hcc_gt_pred")


## Figure 2c, continued. AUROC and composition on the same HCC section

同一张 `kqr-83230_aligned_3bba12d6`：上排是 L1 / L12 / L2 的 one-versus-rest ROC
（概率来自 `stardist_roc/HCC__kqr-83230_aligned_3bba12d6/`，规则同 `macro_auc_ovr`）；
下排是该切片上 **GT vs Pred** 的 100% 堆叠构成，便于对照精度和成分偏移。
（独立的 class-wise ROC 仍见下面的 Figure 2d。）


In [ ]:
ROC_DIR = panels.stardist_roc_dir(DATA, "HCC", hcc_sample)

fig, auc_df, comp_df = panels.plot_auroc_gt_pred_composition(
    adata,
    ROC_DIR,
    TIER_PALETTE,
    macro_fn=macro_auc_ovr,
    figsize_cm=(16.4, 11.2),
    title="HCC kqr-83230 (trained, fold 0)  AUROC and GT vs Pred composition",
    auroc_csv=OUT / "fig2c_hcc_per_class_auroc.csv",
    composition_csv=OUT / "fig2c_hcc_gt_pred_composition.csv",
)
save_fig(fig, "fig2c_hcc_auroc_composition")


## Figure 2c, continued. HCC sections for the immunotherapy contrast

临床表是 `data/CODEX/HCC/Michael_data_transfer/s4769/HE/s4769_he_mapping_updated_Visium.xlsx` 的 `Clinical_info`，用 `MATCHED_HE` 接到切片。`Response` 是应答或无应答，`diagnosis` 是治疗前或治疗后。下面三张和上面同一版式，选的是这篇论文用来区分应答和耐药的结构。

- `kqr-83230_aligned_5b50bc37`：同一病人 85560，无应答，治疗前（区域 73_2，fold 2）。成纤维细胞和中性粒细胞为主，作为上面那张治疗后髓系肿瘤的配对基线（Fig. 5H）。
- `lvr-76315_aligned_9c96ea90`：病人 85566，应答，治疗后（区域 83_4，fold 0）。CD8 T 细胞留在肿瘤旁边，对应 Fig. 5C 的淋巴细胞富集肿瘤。
- `fcu-91631_aligned_4722207f`：病人 85561，无应答，治疗后（区域 74_4，fold 3）。内皮细胞约占 15%，肿瘤团被基质和血管隔开，对应 Fig. 5C 的内皮富集肿瘤，也最接近文中把淋巴聚集和肿瘤隔开的界面。

空间图分别是 `fig2c_hcc_nonresponder_pre_85560`、`fig2c_hcc_responder_post_85566`、`fig2c_hcc_nonresponder_post_endo_85561`。下一格对每一张再画与上面 `fig2c_hcc_auroc_composition` 同版式的 per-class AUROC 和 GT vs Pred 堆积图，文件名加 `_auroc_composition` 后缀。

四张放在一起，要说的是 Hist2Pheno 从 H&E 核上恢复了这篇论文用 CODEX 定义的空间结构，不是模型直接预测了应答。训练标签是七个癌种共用的 L1 / L12 / L2 细胞类型，没有应答或治疗前后这个标签。每张图上排是 CODEX 标注，下排是只看 H&E 的预测。

- 同一无应答病人：治疗前是成纤维细胞和中性粒细胞，治疗后变成巨噬细胞贴着肿瘤。模型没有拿到治疗时间，两张图的 L2 组成已经不同。
- 同是治疗后：应答者的 CD8 T 细胞留在肿瘤旁；一名无应答者是巨噬细胞富集的肿瘤；另一名无应答者的肿瘤团被内皮和基质隔开。这三种就是 Fig. 5 的淋巴细胞富集肿瘤、髓系富集肿瘤和内皮富集肿瘤。
- L1 区分上皮、免疫和基质的地盘；L2 再把免疫拆成 CD8、巨噬细胞和中性粒细胞，并单独标出内皮。论文用来区分应答和耐药的，正是这一层。

这四张在 UNI2 A2 的划分里分别是 fold 0、fold 2、fold 0、fold 3，都不是部署折 fold 1，所以 fold 1 的权重训练时都见过它们。图上的预测是部署结果。看图时对上下两排的大块结构：85561 的肿瘤团和隔开它们的基质，85566 里伸进肿瘤的免疫细胞。


In [ ]:
# Extra HCC sections from the same trial (Wu, Trevino et al., bioRxiv 2025.06.11.656869).
# The cell above is already patient 85560, non-responder, posttreatment (region 73_4, fold 0 under UNI2 A2):
# macrophage-rich tumor, the resistance niche in their Fig. 5.
# These three add the contrasts that paper highlights, using the same GT vs Hist2Pheno layout.
showcase = [
    (
        "kqr-83230_aligned_5b50bc37",
        "Non-responder, pretreatment\npatient 85560, region 73_2",
        "fig2c_hcc_nonresponder_pre_85560",
    ),
    (
        "lvr-76315_aligned_9c96ea90",
        "Responder, posttreatment\npatient 85566, region 83_4",
        "fig2c_hcc_responder_post_85566",
    ),
    (
        "fcu-91631_aligned_4722207f",
        "Non-responder, posttreatment\npatient 85561, region 74_4",
        "fig2c_hcc_nonresponder_post_endo_85561",
    ),
]

# Same visual params as the section above. Predictions are the UNI2 A2 deployment.
POINT_S = 0.4
FIGSIZE_CM = (16.4, 12.6)

for sample, title, stem in showcase:
    adata_s = ad.read_h5ad(panels.matched_pred_h5ad(DATA, "HCC", sample))
    fig = panels.plot_gt_pred_spatial(
        adata_s,
        TIER_PALETTE,
        point_s=POINT_S,
        figsize_cm=FIGSIZE_CM,
        title=title,
    )
    print(sample)
    save_fig(fig, stem)


In [ ]:
# Same layout as fig2c_hcc_auroc_composition, one figure per showcase section.
for sample, title, stem in showcase:
    adata_s = ad.read_h5ad(panels.matched_pred_h5ad(DATA, "HCC", sample))
    fig, auc_df, comp_df = panels.plot_auroc_gt_pred_composition(
        adata_s,
        panels.stardist_roc_dir(DATA, "HCC", sample),
        TIER_PALETTE,
        macro_fn=macro_auc_ovr,
        figsize_cm=(16.4, 11.2),
        title=f"{title.replace(chr(10), ' · ')}  AUROC and GT vs Pred composition",
        auroc_csv=OUT / f"{stem}_per_class_auroc.csv",
        composition_csv=OUT / f"{stem}_gt_pred_composition.csv",
    )
    print(sample)
    save_fig(fig, f"{stem}_auroc_composition")


## Figure 2d. Class-wise ROC on that section

`stardist_roc/` 里的 CSV 是逐核的预测和概率，不是已经算好的 AUC。这里按 `macro_auc_ovr` 的规则重画 one-versus-rest：某一类在该切片上没有阳性或没有阴性就不画。

In [ ]:
ROC_DIR = panels.stardist_roc_dir(DATA, "HCC", hcc_sample)

fig, auc_df = panels.plot_stardist_tier_roc(
    ROC_DIR,
    TIER_PALETTE,
    macro_fn=macro_auc_ovr,
    figsize_cm=(16.2, 5.6),
    legend_fontsize=5,
    auroc_csv=OUT / "fig2d_hcc_per_class_auroc.csv",
)
save_fig(fig, "fig2d_hcc_roc")


## Figure 3. Fields already on disk

每一行四格的顺序是 H&E、细胞核 L2、spot、一个 niche。这一格只导出已经画好的 H&E、L2 和 niche。

- L2 取空间图的右栏。
- Niche 取三栏图的中间栏，也就是半径 55 µm 的空间得分。
- HCC 用 `P1_N` 和 `P1_T` 的 SRI。
- GBM 用 `P174511_Initial` 接触表最下面一行的 L2，也就是全部 StarDist 核。
- CRC 的 H&E 是 11 GB BigTIFF，这里不解码。

这些空间 PNG 只从 UNI2 A2 的 `pred_uni2_A2_bestmlp_paired_v2_seed42/figures/` 裁。文件还没生成时，这一格留空。GBM 的 `P174511_Initial` 在 UNI2 划分里是 fold 0，接触表是部署预测，不是留出切片。


In [ ]:
def export_he(src, dest):
    if src is None or not Path(src).is_file():
        print("H&E left blank:", dest.name)
        return None
    fig23.he_thumb(src, dest, max_px=900)
    print(dest)
    return dest

fields = []

def add_field(letter, cancer, kind, path):
    fields.append({"letter": letter, "cancer": cancer, "field": kind, "path": str(path) if path else ""})

specs = [
    ("a", "Prostate",
     DATA / "Visium/Prostate/Patient_1/HE_images/H1_2.jpg",
     DATA / "Visium/Prostate" / EXT_PRED / "figures/Patient_1__H1_2_spatial_L1_L12_L2.png",
     DATA / "Visium/Prostate" / EXT_PRED / "figures/Patient_1__H1_2_TLS_abundance_vs_spatial.png",
     "tls"),
    ("b", "NPC",
     DATA / "Visium/NPC/HE_images/20210809-C-AH4199551.tif",
     DATA / "Visium/NPC" / EXT_PRED / "figures/20210809-C-AH4199551_spatial_L1_L12_L2.png",
     DATA / "Visium/NPC" / EXT_PRED / "figures/20210809-C-AH4199551_TLS_abundance_vs_spatial.png",
     "tls"),
    ("c_N", "HCC P1_N",
     DATA / "Visium/HCC/HE_images/P1_N.jpg",
     DATA / "Visium/HCC" / EXT_PRED / "figures/P1_N_spatial_L1_L12_L2.png",
     DATA / "Visium/HCC" / EXT_PRED / "figures/P1_N_SRI_abundance_vs_spatial.png",
     "sri"),
    ("c_T", "HCC P1_T",
     DATA / "Visium/HCC/HE_images/P1_T.jpg",
     DATA / "Visium/HCC" / EXT_PRED / "figures/P1_T_spatial_L1_L12_L2.png",
     DATA / "Visium/HCC" / EXT_PRED / "figures/P1_T_SRI_abundance_vs_spatial.png",
     "sri"),
    ("d", "Breast",
     DATA / "ST/BRCA/HE_images/10426_HE_BT_C1.jpg",
     DATA / "ST/BRCA" / EXT_PRED / "figures/10426_HE_BT_C1_spatial_L1_L12_L2.png",
     DATA / "ST/BRCA" / EXT_PRED / "figures/10426_HE_BT_C1_TLS_abundance_vs_spatial.png",
     "tls"),
    ("e", "CRC",
     None,
     DATA / "Visium/CRC" / EXT_PRED / "figures/Visium_V2_Human_Colon_Cancer_P2_tissue_image_spatial_L1_L12_L2.png",
     DATA / "Visium/CRC" / EXT_PRED / "figures/Visium_V2_Human_Colon_Cancer_P2_tissue_image_TLS_abundance_vs_spatial.png",
     "tls"),
]

for letter, cancer, he, spatial, niche, niche_name in specs:
    he_out = OUT / f"fig3{letter}_he.jpg"
    l2_out = OUT / f"fig3{letter}_l2.png"
    niche_out = OUT / f"fig3{letter}_{niche_name}.png"
    export_he(he, he_out)
    if spatial is not None and Path(spatial).is_file():
        fig23.l2_third(spatial, l2_out)
        print(l2_out)
    else:
        print("UNI2 spatial panel not saved yet:", spatial)
        l2_out = None
    if niche is not None and Path(niche).is_file():
        fig23.niche_middle(niche, niche_out)
        print(niche_out)
    else:
        print("UNI2 niche panel not saved yet:", niche)
        niche_out = None
    add_field(letter, cancer, "H&E", he_out if he else None)
    add_field(letter, cancer, "L2", l2_out)
    add_field(letter, cancer, niche_name, niche_out)

gbm_he = (
    DATA / "CODEX/GBM/WangLab/4_Images/CytAssist_Image"
    / "P174511_2_3_2024-05-11_14-33-44_2024-05-11_14-05-32_H1-8BZDCBG_A1_sample.tif"
)
gbm_panel = (
    UNI2_PRED / "matched/GBM/P174511_Initial"
    / "P174511_Initial_panel_4x3.png"
)
export_he(gbm_he, OUT / "fig3f_he.jpg")
if gbm_panel.is_file():
    fig23.gbm_l2(gbm_panel, OUT / "fig3f_l2.png")
    print(OUT / "fig3f_l2.png")
    gbm_l2_path = OUT / "fig3f_l2.png"
else:
    print("UNI2 GBM contact sheet not saved yet:", gbm_panel)
    gbm_l2_path = None
add_field("f", "GBM", "H&E", OUT / "fig3f_he.jpg")
add_field("f", "GBM", "L2", gbm_l2_path)
print("CRC H&E source is Visium_V2_Human_Colon_Cancer_P2_tissue_image.btf (11 GB). Drop a JPEG into this folder when it is ready.")


## Figure 3 spot composition

预测来自 UNI2 A2 best model，文件在各队列的 `pred_uni2_A2_bestmlp_paired_v2_seed42/`。GBM 全核图在训练部署目录的 `stardist_all/`。`P174511_Initial` 是 fold 0，这张图是部署预测。

细胞核和 spot 画在同一套全分辨率像素上，再按 Li et al., *npj Precision Oncology* (2025) 的 `CTA_align` 把分类收进捕获区。

`CTA_align` 的步骤是：SpaceRanger 的 spot 中心除以 `scaling_factor`，回到显微镜全分辨率；捕获半径是 spot 直径的一半，再除以同一个缩放系数；QuPath 质心若以 µm 存储，除以 µm/px 变成像素；Visium 和 QuPath 的 y 轴相反时，用图像高度做一次翻转。质心落在以 spot 为中心、半边长等于捕获半径的正方形里，才算被这个 spot 捕获；方框外记为 `Not_captured`。每个 spot 内按硬分类计数，再除以该 spot 的核数。

这里的核坐标已经在 H&E 全分辨率上，所以 `scaling_factor = 1`，也不做第二次 y 轴翻转。捕获区用圆盘而不是那个正方形：孔是圆的，半径仍是直径的一半。孔心距大于直径时，一个核至多落入一个圆。颜色是该圆内核数最多的 L2。

- Breast `10426_HE_BT_C1` 对应 HER2ST `D1`。ST 阵列的孔直径是 100 µm，孔心距 200 µm，所以半径是测到的孔心距的 1/4。坐标在 `ST-spotfiles/D1_selection.tsv` 的 `pixel_x`、`pixel_y`，与 `spatial_HE` 同一像素系。
- GBM `P174511_Initial` 用 16 µm bin。bin 铺满组织，用边长等于 bin 的正方形，和 `CTA_align` 的方框同一写法。这张切片没有 softmax，颜色是 bin 内的多数类。
- NPC patient 1 是 `20210809-C-AH4199551`。spot 坐标在 `data/Visium/NPC/patient1/tissue_positions_list.csv`，只用 `in_tissue == 1`。`pxl_col_in_fullres`、`pxl_row_in_fullres` 与 `spatial_HE` 同一像素系，不翻转 y 轴。捕获半径是 `scalefactors_json.json` 里 `spot_diameter_fullres` 的一半。左边是单核 L2，右边是这些 Visium spot 的多数类。
- NPC patient 2 是 `20210809-D-AJ3037946`，坐标在 `data/Visium/NPC/patient2/`，画法与 patient 1 相同。
- Prostate `Patient_1__H2_5` 画法与 NPC 相同。spot 坐标在 `data/Visium/Prostate/Patient_1/Visium_with_annotation/H2_5/`（`tissue_positions_list.csv` 与 `scalefactors_json.json`）。左边是单核 L2，右边是 Visium spot 的多数类。
- HCC、CRC 没有可用的 spot 坐标，不画。


In [ ]:
import json
import anndata as ad
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.spatial import cKDTree

L2_NAMES = list(L2_PALETTE)


def legend_handles(names):
    return [
        Line2D(
            [0], [0], marker="o", color="none", markerfacecolor=L2_PALETTE[name],
            markeredgecolor="none", markersize=4, label=name,
        )
        for name in names
    ]


def draw_spots(xy, dominant, pitch_px, title, stem, counts, marker_radius=None, width_cm=7.4, height_cm=7.2, marker_s=2):
    keep = counts > 0
    xy = np.asarray(xy)[keep]
    dominant = np.asarray(dominant)[keep]
    fig, ax = plt.subplots(figsize=(width_cm / 2.54, height_cm / 2.54))
    present = [name for name in L2_PALETTE if np.any(dominant == name)]
    radius = 0.45 * float(pitch_px) if marker_radius is None else float(marker_radius)
    if len(xy) > 4000:
        for name in present:
            mask = dominant == name
            ax.scatter(xy[mask, 0], xy[mask, 1], s=marker_s, c=[L2_PALETTE[name]], linewidths=0, rasterized=True)
    else:
        patches, colors = [], []
        for name in present:
            for x, y in xy[dominant == name]:
                patches.append(Circle((x, y), radius))
                colors.append(L2_PALETTE[name])
        ax.add_collection(PatchCollection(patches, facecolors=colors, edgecolors="none"))
    pad = float(pitch_px)
    ax.set_xlim(xy[:, 0].min() - pad, xy[:, 0].max() + pad)
    ax.set_ylim(xy[:, 1].max() + pad, xy[:, 1].min() - pad)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    ax.legend(handles=legend_handles(present), frameon=False, fontsize=5, loc="center left", bbox_to_anchor=(1.02, 0.5))
    save_fig(fig, stem)


In [ ]:
sample = "10426_HE_BT_C1"
pred_path = uni2_external_pred("ST/BRCA", sample)
stored = np.load(pred_path.with_name(f"{sample}_l2_probs.npz"), allow_pickle=True)
adata = ad.read_h5ad(pred_path)
nuc_xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
probs = np.asarray(stored["probs"], dtype=np.float32)
class_names = [str(x) for x in stored["class_names"].tolist()]
if set(class_names) != set(L2_NAMES):
    raise RuntimeError("L2 names in the npz are not the pan-cancer palette")
if probs.shape[0] != nuc_xy.shape[0]:
    raise RuntimeError(f"probs {probs.shape[0]} != nuclei {nuc_xy.shape[0]}")
spots = pd.read_csv(DATA / "ST/BRCA/her2st/data/ST-spotfiles/D1_selection.tsv", sep="\t")
spots = spots.loc[spots["selected"] == 1].reset_index(drop=True)
spot_xy = spots[["pixel_x", "pixel_y"]].to_numpy(dtype=np.float64)
pitch_px = float(np.median(cKDTree(spot_xy).query(spot_xy, k=2)[0][:, 1]))
radius_px = 0.25 * pitch_px
hard = np.array(class_names)[probs.argmax(1)]
captured = nuclei_to_spots(
    nuc_xy, spot_xy, radius_px, shape="circle",
    labels=hard, class_names=class_names, values=probs,
)
counts = captured["n_nuclei"]
breast = spots.copy()
breast["n_nuclei"] = counts
breast["dominant_l2"] = captured["dominant"]
breast["dominant_mean_prob"] = np.where(counts > 0, captured["mean"].max(1), np.nan)
for i, name in enumerate(class_names):
    breast[f"mean_{name}"] = captured["mean"][:, i]
breast.to_csv(OUT / "fig3d_breast_spot_composition.csv", index=False)
print(
    f"Breast spots {len(spot_xy)}, pitch {pitch_px:.1f} px, radius {radius_px:.1f} px, "
    f"nuclei assigned {int(captured['inside'].sum())} / {len(nuc_xy)}, spots with nuclei {(counts > 0).sum()}"
)
draw_spots(
    spot_xy, captured["dominant"], pitch_px,
    "HER2ST D1    100 µm spots    majority L2",
    "fig3d_breast_spot_l2", counts, marker_radius=2*radius_px,
    width_cm=4.4, height_cm=4.2,
)


In [ ]:

nucleus_s = 0.05
present = [name for name in class_names if np.any(hard == name)]
fig, ax = plt.subplots(figsize=(4.4 / 2.54, 4.2 / 2.54))
for name in class_names:
    mask = hard == name
    if np.any(mask):
        ax.scatter(
            nuc_xy[mask, 0], nuc_xy[mask, 1], s=nucleus_s,
            c=[L2_PALETTE[name]], linewidths=0, rasterized=True,
        )
pad = pitch_px
ax.set_xlim(nuc_xy[:, 0].min() - pad, nuc_xy[:, 0].max() + pad)
ax.set_ylim(nuc_xy[:, 1].max() + pad, nuc_xy[:, 1].min() - pad)
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("HER2ST D1    single nuclei")
ax.legend(
    handles=legend_handles(present), frameon=False, fontsize=5,
    loc="center left", bbox_to_anchor=(1.02, 0.5),
)
save_fig(fig, "fig3d_breast_nuclei")

In [ ]:

gbm_h5 = (
    UNI2_PRED / "stardist_all/GBM/P174511_Initial"
    / "P174511_Initial_all_stardist_pred.h5ad"
)
pos_path = DATA / "CODEX/GBM/WangLab/1_Bin_Matrix/P174511_Initial/square_016um/spatial/tissue_positions.parquet"
scale = json.loads(pos_path.with_name("scalefactors_json.json").read_text())
bin_px = float(scale["spot_diameter_fullres"])
gbm = ad.read_h5ad(gbm_h5)
gbm_xy = np.asarray(gbm.obsm["spatial_HE"], dtype=np.float64)
gbm_lab = gbm.obs["pred_CT"].astype(str).to_numpy()
pos = pd.read_parquet(pos_path)
pos = pos.loc[pos["in_tissue"] == 1].reset_index(drop=True)
bin_xy = pos[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy(dtype=np.float64)
captured = nuclei_to_spots(
    gbm_xy, bin_xy, 0.5 * bin_px, shape="square",
    labels=gbm_lab, class_names=L2_NAMES,
)
counts = captured["n_nuclei"]
used = counts > 0
fraction = np.divide(
    captured["class_counts"].max(1), counts, out=np.zeros(len(counts)), where=used,
)
out = pd.DataFrame({
    "barcode": pos.loc[used, "barcode"].to_numpy(),
    "pixel_x": bin_xy[used, 0],
    "pixel_y": bin_xy[used, 1],
    "n_nuclei": counts[used],
    "dominant_l2": captured["dominant"][used],
    "dominant_fraction": fraction[used],
})
out.to_csv(OUT / "fig3f_gbm_bin16_composition.csv", index=False)
print(
    f"GBM bins in tissue {len(bin_xy)}, pitch {bin_px:.2f} px ({scale['bin_size_um']} µm), "
    f"nuclei inside a bin {int(captured['inside'].sum())} / {len(gbm_xy)}, bins with nuclei {int(used.sum())}"
)
draw_spots(
    bin_xy, captured["dominant"], bin_px,
    "P174511_Initial    16 µm bins    majority L2 label",
    "fig3f_gbm_bin16_l2", counts,
    width_cm=8.0, height_cm=8.0, marker_s=0.35,
)


In [ ]:
sample = "20210809-C-AH4199551"
npc_dir = DATA / "Visium/NPC/patient1"
scale = json.loads((npc_dir / "scalefactors_json.json").read_text())
radius_px = float(scale["spot_diameter_fullres"]) / 2
pos = pd.read_csv(
    npc_dir / "tissue_positions_list.csv",
    header=None,
    names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"],
)
pos = pos.loc[pos["in_tissue"] == 1].reset_index(drop=True)
spot_xy = pos[["pxl_col", "pxl_row"]].to_numpy(dtype=np.float64)
pitch_px = float(np.median(cKDTree(spot_xy).query(spot_xy, k=2)[0][:, 1]))
npc = ad.read_h5ad(uni2_external_pred("Visium/NPC", sample))
npc_xy = np.asarray(npc.obsm["spatial_HE"], dtype=np.float64)
npc_lab = npc.obs["pred_CT"].astype(str).to_numpy()
captured = nuclei_to_spots(
    npc_xy, spot_xy, radius_px, shape="circle",
    labels=npc_lab, class_names=L2_NAMES,
)
counts = captured["n_nuclei"]
used = counts > 0
npc_spots = pos.loc[used].copy()
npc_spots["pixel_x"] = spot_xy[used, 0]
npc_spots["pixel_y"] = spot_xy[used, 1]
npc_spots["n_nuclei"] = counts[used]
npc_spots["dominant_l2"] = captured["dominant"][used]
for i, name in enumerate(L2_NAMES):
    npc_spots[f"n_{name}"] = captured["class_counts"][used, i]
    npc_spots[f"fraction_{name}"] = captured["class_counts"][used, i] / counts[used]
npc_spots.to_csv(OUT / "fig3b_npc_visium_composition.csv", index=False)
nuc_frac = pd.Series(npc_lab).value_counts(normalize=True)
spot_frac = npc_spots["dominant_l2"].value_counts(normalize=True)
present = [name for name in L2_NAMES if name in set(nuc_frac.index) or name in set(spot_frac.index)]
frac = pd.DataFrame({
    "L2": present,
    "single_nucleus": [float(nuc_frac.get(name, 0.0)) for name in present],
    "visium_majority": [float(spot_frac.get(name, 0.0)) for name in present],
})
frac.to_csv(OUT / "fig3b_npc_class_fraction.csv", index=False)
display(frac)
print(
    f"NPC {sample}: nuclei {len(npc_xy):,}, in-tissue spots {len(spot_xy):,}, "
    f"pitch {pitch_px:.1f} px, capture radius {radius_px:.1f} px, "
    f"nuclei in a spot {int(captured['inside'].sum()):,} / {len(npc_xy):,}, "
    f"spots with nuclei {int(used.sum()):,}"
)

nucleus_s = 0.005
spot_draw_radius = 0.45 * pitch_px

fig, axes = plt.subplots(1, 2, figsize=(8 / 2.54, 4 / 2.54))
axes[0].set_title("Single nuclei")
for name in L2_NAMES:
    mask = npc_lab == name
    if np.any(mask):
        axes[0].scatter(npc_xy[mask, 0], npc_xy[mask, 1], s=nucleus_s,
                        c=[L2_PALETTE[name]], linewidths=0, rasterized=True)
spot_draw = spot_xy[used]
spot_lab = captured["dominant"][used]
patches, colors = [], []
for name in L2_NAMES:
    if not np.any(spot_lab == name):
        continue
    for x, y in spot_draw[spot_lab == name]:
        patches.append(Circle((x, y), spot_draw_radius))
        colors.append(L2_PALETTE[name])
axes[1].add_collection(PatchCollection(patches, facecolors=colors, edgecolors="none"))
axes[1].set_title("Visium spots")
pad = pitch_px
for ax in axes:
    ax.set_xlim(npc_xy[:, 0].min() - pad, npc_xy[:, 0].max() + pad)
    ax.set_ylim(npc_xy[:, 1].max() + pad, npc_xy[:, 1].min() - pad)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
axes[1].legend(handles=legend_handles(present), frameon=False, fontsize=5, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.suptitle(f"NPC {sample}", fontsize=7, y=1.02)
save_fig(fig, "fig3b_npc_nuclei_vs_visium")

skipped = pd.DataFrame([
    {"letter": "c", "cancer": "HCC", "reason": "No spatial/ directory for the P1_N and P1_T JPEGs"},
    {"letter": "e", "cancer": "CRC", "reason": "No tissue_positions in data/Visium/CRC"},
])
skipped.to_csv(OUT / "fig3_spot_maps_not_drawn.csv", index=False)
display(skipped)


In [ ]:
sample = "20210809-D-AJ3037946"
npc_dir = DATA / "Visium/NPC/patient2"
scale = json.loads((npc_dir / "scalefactors_json.json").read_text())
radius_px = float(scale["spot_diameter_fullres"]) / 2
pos = pd.read_csv(
    npc_dir / "tissue_positions_list.csv",
    header=None,
    names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"],
)
pos = pos.loc[pos["in_tissue"] == 1].reset_index(drop=True)
spot_xy = pos[["pxl_col", "pxl_row"]].to_numpy(dtype=np.float64)
pitch_px = float(np.median(cKDTree(spot_xy).query(spot_xy, k=2)[0][:, 1]))
npc = ad.read_h5ad(uni2_external_pred("Visium/NPC", sample))
npc_xy = np.asarray(npc.obsm["spatial_HE"], dtype=np.float64)
npc_lab = npc.obs["pred_CT"].astype(str).to_numpy()
captured = nuclei_to_spots(
    npc_xy, spot_xy, radius_px, shape="circle",
    labels=npc_lab, class_names=L2_NAMES,
)
counts = captured["n_nuclei"]
used = counts > 0
npc_spots = pos.loc[used].copy()
npc_spots["pixel_x"] = spot_xy[used, 0]
npc_spots["pixel_y"] = spot_xy[used, 1]
npc_spots["n_nuclei"] = counts[used]
npc_spots["dominant_l2"] = captured["dominant"][used]
for i, name in enumerate(L2_NAMES):
    npc_spots[f"n_{name}"] = captured["class_counts"][used, i]
    npc_spots[f"fraction_{name}"] = captured["class_counts"][used, i] / counts[used]
npc_spots.to_csv(OUT / "fig3b_npc_p2_visium_composition.csv", index=False)
nuc_frac = pd.Series(npc_lab).value_counts(normalize=True)
spot_frac = npc_spots["dominant_l2"].value_counts(normalize=True)
present = [name for name in L2_NAMES if name in set(nuc_frac.index) or name in set(spot_frac.index)]
frac = pd.DataFrame({
    "L2": present,
    "single_nucleus": [float(nuc_frac.get(name, 0.0)) for name in present],
    "visium_majority": [float(spot_frac.get(name, 0.0)) for name in present],
})
frac.to_csv(OUT / "fig3b_npc_p2_class_fraction.csv", index=False)
display(frac)
print(
    f"NPC {sample}: nuclei {len(npc_xy):,}, in-tissue spots {len(spot_xy):,}, "
    f"pitch {pitch_px:.1f} px, capture radius {radius_px:.1f} px, "
    f"nuclei in a spot {int(captured['inside'].sum()):,} / {len(npc_xy):,}, "
    f"spots with nuclei {int(used.sum()):,}"
)

nucleus_s = 0.005
spot_draw_radius = 0.45 * pitch_px

fig, axes = plt.subplots(1, 2, figsize=(8 / 2.54, 4 / 2.54))
axes[0].set_title("Single nuclei")
for name in L2_NAMES:
    mask = npc_lab == name
    if np.any(mask):
        axes[0].scatter(npc_xy[mask, 0], npc_xy[mask, 1], s=nucleus_s,
                        c=[L2_PALETTE[name]], linewidths=0, rasterized=True)
spot_draw = spot_xy[used]
spot_lab = captured["dominant"][used]
patches, colors = [], []
for name in L2_NAMES:
    if not np.any(spot_lab == name):
        continue
    for x, y in spot_draw[spot_lab == name]:
        patches.append(Circle((x, y), spot_draw_radius))
        colors.append(L2_PALETTE[name])
axes[1].add_collection(PatchCollection(patches, facecolors=colors, edgecolors="none"))
axes[1].set_title("Visium spots")
pad = pitch_px
for ax in axes:
    ax.set_xlim(npc_xy[:, 0].min() - pad, npc_xy[:, 0].max() + pad)
    ax.set_ylim(npc_xy[:, 1].max() + pad, npc_xy[:, 1].min() - pad)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
axes[1].legend(handles=legend_handles(present), frameon=False, fontsize=5, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.suptitle(f"NPC {sample}", fontsize=7, y=1.02)
save_fig(fig, "fig3b_npc_p2_nuclei_vs_visium")


In [ ]:
sample = "Patient_1__H2_5"
# Mendeley svw96g68dv: Patient 1 / Visium_with_annotation / H2_5
spot_dir = DATA / "Visium/Prostate/Patient_1/Visium_with_annotation/H2_5"
pos_path = spot_dir / "tissue_positions_list.csv"
if not pos_path.is_file():
    alt = spot_dir / "tissue_positions.csv"
    pos_path = alt if alt.is_file() else None

pr = ad.read_h5ad(uni2_external_pred("Visium/Prostate", sample))
pr_xy = np.asarray(pr.obsm["spatial_HE"], dtype=np.float64)
pr_lab = pr.obs["pred_CT"].astype(str).to_numpy()
nuc_frac = pd.Series(pr_lab).value_counts(normalize=True)

if pos_path is None:
    present = [name for name in L2_NAMES if name in set(nuc_frac.index)]
    frac = pd.DataFrame({
        "L2": present,
        "single_nucleus": [float(nuc_frac.get(name, 0.0)) for name in present],
    })
    frac.to_csv(OUT / "fig3a_prostate_h2_5_class_fraction.csv", index=False)
    display(frac)
    print(
        f"Prostate {sample}: nuclei {len(pr_xy):,}. "
        f"No tissue_positions under {spot_dir}; Visium spots are not drawn."
    )
    nucleus_s = 0.005
    fig, ax = plt.subplots(figsize=(4 / 2.54, 4 / 2.54))
    for name in L2_NAMES:
        mask = pr_lab == name
        if np.any(mask):
            ax.scatter(pr_xy[mask, 0], pr_xy[mask, 1], s=nucleus_s,
                       c=[L2_PALETTE[name]], linewidths=0, rasterized=True)
    pad = 40
    ax.set_xlim(pr_xy[:, 0].min() - pad, pr_xy[:, 0].max() + pad)
    ax.set_ylim(pr_xy[:, 1].max() + pad, pr_xy[:, 1].min() - pad)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Prostate Patient_1 H2_5\nsingle nuclei", loc="left")
    ax.legend(handles=legend_handles(present), frameon=False, fontsize=5, loc="center left", bbox_to_anchor=(1.02, 0.5))
    save_fig(fig, "fig3a_prostate_h2_5_nuclei")
else:
    scale = json.loads((spot_dir / "scalefactors_json.json").read_text())
    radius_px = float(scale["spot_diameter_fullres"]) / 2
    header = pd.read_csv(pos_path, nrows=0).columns.tolist()
    if "barcode" in header:
        pos = pd.read_csv(pos_path)
        pos = pos.rename(columns={"pxl_row_in_fullres": "pxl_row", "pxl_col_in_fullres": "pxl_col"})
    else:
        pos = pd.read_csv(
            pos_path, header=None,
            names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"],
        )
    pos = pos.loc[pos["in_tissue"] == 1].reset_index(drop=True)
    spot_xy = pos[["pxl_col", "pxl_row"]].to_numpy(dtype=np.float64)
    pitch_px = float(np.median(cKDTree(spot_xy).query(spot_xy, k=2)[0][:, 1]))
    captured = nuclei_to_spots(
        pr_xy, spot_xy, radius_px, shape="circle",
        labels=pr_lab, class_names=L2_NAMES,
    )
    counts = captured["n_nuclei"]
    used = counts > 0
    pr_spots = pos.loc[used].copy()
    pr_spots["pixel_x"] = spot_xy[used, 0]
    pr_spots["pixel_y"] = spot_xy[used, 1]
    pr_spots["n_nuclei"] = counts[used]
    pr_spots["dominant_l2"] = captured["dominant"][used]
    for i, name in enumerate(L2_NAMES):
        pr_spots[f"n_{name}"] = captured["class_counts"][used, i]
        pr_spots[f"fraction_{name}"] = captured["class_counts"][used, i] / counts[used]
    pr_spots.to_csv(OUT / "fig3a_prostate_h2_5_visium_composition.csv", index=False)
    spot_frac = pr_spots["dominant_l2"].value_counts(normalize=True)
    present = [name for name in L2_NAMES if name in set(nuc_frac.index) or name in set(spot_frac.index)]
    frac = pd.DataFrame({
        "L2": present,
        "single_nucleus": [float(nuc_frac.get(name, 0.0)) for name in present],
        "visium_majority": [float(spot_frac.get(name, 0.0)) for name in present],
    })
    frac.to_csv(OUT / "fig3a_prostate_h2_5_class_fraction.csv", index=False)
    display(frac)
    print(
        f"Prostate {sample}: nuclei {len(pr_xy):,}, in-tissue spots {len(spot_xy):,}, "
        f"pitch {pitch_px:.1f} px, capture radius {radius_px:.1f} px, "
        f"nuclei in a spot {int(captured['inside'].sum()):,} / {len(pr_xy):,}, "
        f"spots with nuclei {int(used.sum()):,}"
    )
    nucleus_s = 0.005
    spot_draw_radius = 0.45 * pitch_px
    fig, axes = plt.subplots(1, 2, figsize=(8 / 2.54, 4 / 2.54))
    axes[0].set_title("Single nuclei")
    for name in L2_NAMES:
        mask = pr_lab == name
        if np.any(mask):
            axes[0].scatter(pr_xy[mask, 0], pr_xy[mask, 1], s=nucleus_s,
                            c=[L2_PALETTE[name]], linewidths=0, rasterized=True)
    spot_draw = spot_xy[used]
    spot_lab = captured["dominant"][used]
    patches, colors = [], []
    for name in L2_NAMES:
        if not np.any(spot_lab == name):
            continue
        for x, y in spot_draw[spot_lab == name]:
            patches.append(Circle((x, y), spot_draw_radius))
            colors.append(L2_PALETTE[name])
    axes[1].add_collection(PatchCollection(patches, facecolors=colors, edgecolors="none"))
    axes[1].set_title("Visium spots")
    pad = pitch_px
    for ax in axes:
        ax.set_xlim(pr_xy[:, 0].min() - pad, pr_xy[:, 0].max() + pad)
        ax.set_ylim(pr_xy[:, 1].max() + pad, pr_xy[:, 1].min() - pad)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[1].legend(handles=legend_handles(present), frameon=False, fontsize=5, loc="center left", bbox_to_anchor=(1.02, 0.5))
    fig.suptitle(f"Prostate {sample}", fontsize=7, y=1.02)
    save_fig(fig, "fig3a_prostate_h2_5_nuclei_vs_visium")


## What was written

上色用的是 `plotting_palettes.get_palette("pancancer")`。这台机器没有 Arial，图里用 Liberation Sans，字号按 7 pt。

In [ ]:
written = sorted(p.name for p in OUT.iterdir() if p.is_file())
print(f"{len(written)} files in {OUT}")
for name in written:
    print(" ", name)